# 10 - CASMI v2 Colab setup

Environment check only: Drive, repository, PYTHONPATH, GPU / CUDA / VRAM, RAM / disk, persistent Drive folders, seeds, AMP choice, configuration, lightweight import checks.

**Nothing is trained here.** Run order: `10 -> 11 -> 12 -> 13 -> 14` (see `docs/CASMI_V2_COLAB_RUNBOOK.md`).

In [ ]:
# --- parameters (edit) -------------------------------------------------------------
DRIVE_ROOT_DEFAULT = '/content/drive/MyDrive/EnvedaCASMI'   # or set ENVEDA_DRIVE_ROOT before running
REPO_GIT_URL = ''          # optional: a git URL to clone into <drive_root>/repo; '' = use the uploaded repo folder
REPO_GIT_BRANCH = 'main'
PULL_IF_GIT_REPO = False   # git pull an existing clone
INSTALL_RDKIT = True       # Colab has no RDKit preinstalled (needed by notebooks 12 / 14)

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path(os.environ.get('ENVEDA_DRIVE_ROOT', DRIVE_ROOT_DEFAULT))
REPO_ROOT = Path(os.environ.get('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
os.environ['ENVEDA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['ENVEDA_REPO_ROOT'] = str(REPO_ROOT)
print('drive root:', DRIVE_ROOT, '| exists:', DRIVE_ROOT.exists())
print('repo root :', REPO_ROOT, '| exists:', REPO_ROOT.exists())

In [ ]:
# repository: clone / pull OR use the uploaded folder
if REPO_GIT_URL and not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_GIT_BRANCH, REPO_GIT_URL, str(REPO_ROOT)], check=True)
elif PULL_IF_GIT_REPO and (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

required = [REPO_ROOT / 'src' / 'casmi' / '__init__.py', REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'repository incomplete -- upload the repo to {REPO_ROOT} or set REPO_GIT_URL. Missing: {missing}')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('src on sys.path:', REPO_ROOT / 'src')

In [ ]:
from casmi.workspace.environment import ensure_packages
spec = {'yaml': 'pyyaml', 'pyarrow': 'pyarrow', 'lightgbm': 'lightgbm', 'psutil': 'psutil'}
if INSTALL_RDKIT:
    spec['rdkit'] = 'rdkit'
print('installed now:', ensure_packages(spec) or 'nothing (all present)')

In [ ]:
import json
from casmi.workspace.config import load_v2_config
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()   # creates every persistent Drive folder (+ /content scratch)
print(json.dumps(P.as_dict(), indent=2))

In [ ]:
from casmi.workspace.environment import gpu_report, system_report, choose_amp_dtype, set_seeds, recommend_batch_size
GPU = gpu_report()
SYS = system_report()
print(json.dumps(GPU, indent=2))
print(json.dumps(SYS, indent=2))
if not GPU['cuda_available']:
    print('NOTE: no CUDA GPU in this runtime. Notebooks 11-14 run on CPU; deep-learning notebooks (20+) require CUDA.')

In [ ]:
SEED = set_seeds(CFG['runtime']['seed'], deterministic=CFG['runtime']['deterministic'])
AMP_DTYPE = choose_amp_dtype(CFG['runtime']['amp'], GPU)
print('seed:', SEED)
print('AMP dtype:', AMP_DTYPE, '(bf16 only where the GPU supports it; fp16 otherwise; None on CPU)')
print('example conservative batch sizes (override always allowed):',
      {t: recommend_batch_size(t, GPU) for t in ('fingerprint', 'contrastive', 'default')})

In [ ]:
# input files the later notebooks need (uploaded by the USER to Drive -- see the runbook)
from casmi.workspace.config import input_path
rows = []
for name in CFG['inputs']:
    p = input_path(CFG, P, name)
    rows.append((name, str(p), p.exists()))
for extra in ('config.json', 'ref_meta.parquet', 'ref_peaks_mz.npy', 'connectivities.parquet'):
    rows.append(('bundle/' + extra, str(P.bundle_dir / extra), (P.bundle_dir / extra).exists()))
for name, path, ok in rows:
    print(('OK      ' if ok else 'MISSING '), name, '->', path)

In [ ]:
# lightweight import checks only (no data is loaded, nothing is computed)
import importlib
modules = ['casmi.validation.regimes', 'casmi.validation.metrics', 'casmi.validation.reporting',
           'casmi.candidates.universe', 'casmi.candidates.mass_index', 'casmi.candidates.formula_index',
           'casmi.candidates.recall', 'casmi.analog.retrieval', 'casmi.analog.propagation', 'casmi.analog.features',
           'casmi.chemistry.fingerprint_store', 'casmi.viz.plots_v2', 'casmi.workspace.experiments']
status = {}
for m in modules:
    try:
        importlib.import_module(m)
        status[m] = 'ok'
    except Exception as e:
        status[m] = f'FAILED: {type(e).__name__}: {e}'
for m, s in status.items():
    print(f'{s:8s} {m}')

In [ ]:
from datetime import datetime, timezone
from casmi.workspace.artifacts import git_commit
env = {'created_by_notebook': '10_colab_v2_setup', 'created_at': datetime.now(timezone.utc).isoformat(),
       'git_commit': git_commit(REPO_ROOT), 'gpu': GPU, 'system': SYS, 'amp_dtype': AMP_DTYPE, 'seed': SEED,
       'paths': P.as_dict(), 'imports': status}
out = P.reports_dir / 'environment' / f"environment_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(env, indent=2, default=str))
print('saved', out)